# Fraud Detection: Final Model and Zindi Submission

Issue 18. We train the final model on **all labelled clients** and predict the **Zindi test set**.

## The final model and why

| Step | Notebook | Result |
|---|---|---|
| Metric | [protocol.md](protocol.md) | **ROC-AUC** (official Zindi metric; ranks clients for STEG's inspections) |
| Model choice | [10_model-comparison.ipynb](10_model-comparison.ipynb) | **LightGBM** had the highest CV ROC-AUC of all default models (0.766 vs 0.750 baseline, 0.738 Random Forest, 0.737 XGBoost) |
| Settings | [11_hyperparameter-tuning.ipynb](11_hyperparameter-tuning.ipynb) | random search, settings saved in `models/lightgbm_best_params.json` |
| Weak spots | [12_error-analysis.ipynb](12_error-analysis.ipynb) | missed fraud with low consumption, long histories, young clients |
| Features | [13_feature-selection.ipynb](13_feature-selection.ipynb) | **feature set v2**: time and meter features added, level max removed, 30 features; CV ROC-AUC 0.769 → **0.848**, validation **0.859** |
| Re-tuning | [14_retuning-new-features.ipynb](14_retuning-new-features.ipynb) | no gain, more overfitting, so not used (team decision 2026-10-22) |

**Final model:** LightGBM, 30 features of feature set v2, settings from notebook 11, fraud class weighted (`class_weight="balanced"`).

**What we do here:**
1. Train on train + validation together (135,493 clients). The validation split has done its job; the extra 20% of clients now helps the final model.
2. Predict a fraud probability for each of the 58,069 Zindi test clients.
3. Write `submission.csv` in the Zindi format and check it.

The notebook shows only tables and numbers, no charts.

In [1]:
# libraries for data handling, the model and the metrics
import json
import pandas as pd
from pathlib import Path
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

pd.set_option("display.max_columns", 40)
DATA_DIR = Path("data")
FEATURE_DIR = DATA_DIR / "features"
OUTPUT_DIR = DATA_DIR / "output"
MODEL_DIR = Path("models")
RANDOM_STATE = 42

## 1. Load feature set v2 and the settings

In [2]:
# read one feature table from data/features, with client_id as the index
def load(name):
    return pd.read_csv(FEATURE_DIR / f"{name}.csv", index_col="client_id")

# the 30 columns of feature set v2 and the settings from notebook 11
features = json.loads((MODEL_DIR / "selected_features.json").read_text())["features"]
params = json.loads((MODEL_DIR / "lightgbm_best_params.json").read_text())

# all labelled clients: the training and the validation split together
X_all = pd.concat([load("X_train_v2"), load("X_valid_v2")])[features]
y_all = pd.concat([load("y_train"), load("y_valid")])["target"].loc[X_all.index]
X_test = load("X_test_v2")[features]

# one row per client, no client in both labelled parts, same columns for test
assert X_all.index.is_unique and list(X_all.columns) == list(X_test.columns)
print("labelled clients:", X_all.shape, f"fraud {y_all.mean():.2%}")
print("test clients:    ", X_test.shape)
params

labelled clients: (135493, 30) fraud 5.58%
test clients:     (58069, 30)


{'colsample_bytree': 0.8676080596203861,
 'learning_rate': 0.039270026903279916,
 'max_depth': 6,
 'min_child_samples': 118,
 'n_estimators': 362,
 'num_leaves': 30,
 'reg_lambda': 1.857328833311487,
 'subsample': 0.822080324639785,
 'class_weight': 'balanced',
 'subsample_freq': 1,
 'random_state': 42}

## 2. Last check: cross-validation on all labelled clients

Before the final fit, we score the final setup once more with 5-fold stratified cross-validation, now on all 135,493 labelled clients. This is the best estimate of the score to expect on the Zindi test set.

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
final_cv = cross_validate(LGBMClassifier(**params, n_jobs=-1, verbose=-1), X_all, y_all, cv=cv,
                          scoring=["roc_auc", "average_precision"])

print(f"CV ROC-AUC: {final_cv['test_roc_auc'].mean():.4f} (± {final_cv['test_roc_auc'].std():.4f})")
print(f"CV PR-AUC:  {final_cv['test_average_precision'].mean():.4f}")

CV ROC-AUC: 0.8524 (± 0.0020)
CV PR-AUC:  0.3115


## 3. Train the final model and save it

In [4]:
# fit on all labelled clients
final_model = LGBMClassifier(**params, n_jobs=-1, verbose=-1).fit(X_all, y_all)

# save the trained trees and a small description of the final setup
MODEL_DIR.mkdir(exist_ok=True)
final_model.booster_.save_model(str(MODEL_DIR / "lightgbm_final.txt"))
(MODEL_DIR / "final_model.json").write_text(json.dumps({
    "model": "LightGBM",
    "features": features,
    "settings": params,
    "trained on": f"{len(X_all)} labelled clients (train + validation)",
    "cv roc_auc": round(float(final_cv["test_roc_auc"].mean()), 4),
}, indent=2))
print(sorted(p.name for p in MODEL_DIR.glob("*final*")))

['final_model.json', 'lightgbm_final.txt']


## 4. Predict the Zindi test set and write the submission

Zindi expects one row per test client with the columns `client_id` and `target` (the fraud probability), in the order of `SampleSubmission.csv`.

The probabilities are higher than the real fraud share because of the class weight. That is fine: ROC-AUC only looks at the **order** of the clients, not at the size of the probabilities.

In [5]:
# fraud probability for every test client, in the order of the sample submission
sample = pd.read_csv(DATA_DIR / "SampleSubmission.csv")
proba = pd.Series(final_model.predict_proba(X_test)[:, 1], index=X_test.index)
submission = pd.DataFrame({"client_id": sample["client_id"], "target": proba.loc[sample["client_id"]].values})

# checks: same clients and order as the sample, one probability per client, all between 0 and 1
assert list(submission.columns) == list(sample.columns)
assert submission["client_id"].equals(sample["client_id"]) and submission["client_id"].is_unique
assert submission["target"].notna().all() and submission["target"].between(0, 1).all()

OUTPUT_DIR.mkdir(exist_ok=True)
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)
print(f"saved {OUTPUT_DIR / 'submission.csv'}: {len(submission)} rows")
submission.head()

saved data/output/submission.csv: 58069 rows


,client_id,target
0,test_Client_0,0.307634
1,test_Client_1,0.757409
2,test_Client_10,0.206389
3,test_Client_100,0.146938
4,test_Client_1000,0.522805


In [6]:
# a quick look at the predicted probabilities (test) next to the training clients
pd.DataFrame({
    "test": submission["target"].describe(),
    "labelled clients": pd.Series(final_model.predict_proba(X_all)[:, 1]).describe(),
}).round(3)

,test,labelled clients
count,58069.000,135493.000
mean,0.310,0.309
std,0.258,0.259
min,0.001,0.000
25%,0.097,0.095
50%,0.238,0.236
75%,0.494,0.491
max,0.983,0.974


## 5. Zindi score

Upload `data/output/submission.csv` on the [Zindi competition page](https://zindi.world/competitions/fraud-detection-in-electricity-and-gas-consumption-challenge) (Submit). The public score is noted here:

| Date | Model | CV ROC-AUC (all labelled clients) | Zindi public score |
|---|---|---|---|
| ZINDI_ROW_PLACEHOLDER |